# Exploratory Data Analysis — AI4I 2020 Predictive Maintenance Dataset

Explores the failure distribution, feature distributions, and correlations that motivate the feature engineering and modeling choices in `src/train.py`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_preprocessing import load_raw_data, clean_columns, engineer_features

df = load_raw_data()
df = clean_columns(df)
df = engineer_features(df)
df.head()

## Failure distribution

The target is heavily imbalanced — this is the central challenge of the project.

In [ ]:
print(df['machine_failure'].value_counts())
print(df['machine_failure'].value_counts(normalize=True))

fig, ax = plt.subplots()
sns.countplot(x='machine_failure', data=df, ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Failure', 'Failure'])
ax.set_title('Failure Distribution')
plt.show()

## Failures by machine type

In [ ]:
fig, ax = plt.subplots()
sns.countplot(x='type', hue='machine_failure', data=df, ax=ax)
ax.set_title('Failures by Machine Type')
plt.show()

## Feature distributions by failure status

Comparing the distribution of each numeric sensor reading for failed vs. non-failed machines.

In [ ]:
numeric_cols = ['air_temp_k', 'process_temp_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'temp_diff_k', 'power_w']
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(data=df, x=col, hue='machine_failure', kde=True, ax=ax, stat='density', common_norm=False)
    ax.set_title(col)
for ax in axes.flat[len(numeric_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## Correlation with failure

In [ ]:
corr_cols = numeric_cols + ['machine_failure']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.show()

corr['machine_failure'].sort_values(ascending=False)

## Takeaways

- The failure rate is ~3.4% — strongly imbalanced, so accuracy alone is a misleading metric (predicting "no failure" for everyone would already score ~96.6% accuracy).
- `torque_nm` and the engineered `power_w` feature show the strongest positive correlation with failure, consistent with overstrain/power-failure mechanics in this dataset.
- `temp_diff_k` (process minus air temperature) is negatively correlated with failure in the raw linear correlation, but still carries useful non-linear signal that the Random Forest model picks up (see `models/metrics.json` `global_importance` for the model's actual learned feature ranking, which can differ from simple linear correlation).